# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [1]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 7


Root proiect: c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [3]:
student_id = "student_02"
handle = "georgesimionoficial"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\data\raw\student_02_youtube_raw.jsonl


In [23]:
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
output_file.parent.mkdir(parents=True, exist_ok=True)

with output_file.open("w", encoding="utf-8") as f:
    for comment in comments:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii raw salvate:", len(comments))
print("Fișier:", output_file)

Comentarii raw salvate: 100
Fișier: c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\data\raw\student_02_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [4]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': '6NWUeLlnMUXaYsBbAPcxLIYK-u0',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': '-vSOTZEwhPOZP60529w2OOrGGM0',
   'id': 'UC5gEyyu61h-BwZfkPlpaJDw'}]}

In [5]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UC5gEyyu61h-BwZfkPlpaJDw'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [6]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': 'eh6NW6b8uFMBWIJPHjugvt_PdhI',
 'id': {'kind': 'youtube#video', 'videoId': '5rHoTX3U_3Q'},
 'snippet': {'publishedAt': '2025-09-30T18:24:37Z',
  'channelId': 'UC5gEyyu61h-BwZfkPlpaJDw',
  'title': 'Turul României: realități de la firul ierbii',
  'description': 'ABONEAZĂ-TE la acest canal de Youtube pentru a afla adevărul din politica românească. CUM TE POȚI ABONA? 1. Apasă pe ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/5rHoTX3U_3Q/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/5rHoTX3U_3Q/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/5rHoTX3U_3Q/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'George Simion',
  'liveBroadcastContent': 'none',
  'publishTime': '2025-09-30T18:24:37Z'}}

In [7]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': '5rHoTX3U_3Q',
  'video_title': 'Turul României: realități de la firul ierbii',
  'video_date': '2025-09-30'},
 {'video_id': 'iH8jB4NlV9Y',
  'video_title': 'Episodul 2: Cum ne-au furat alegerile - Turismul Electoral',
  'video_date': '2025-07-13'},
 {'video_id': 'FrQgc7LoWCU',
  'video_title': 'Jurnalul video al zilei: Sâmbătă dimineață la Gorj în preajmă de Sânziene',
  'video_date': '2025-06-21'},
 {'video_id': 'BGD6hLYnyBk',
  'video_title': '#georgesimion #unitate #democratie #impreuna #prosperitate #românia #gs',
  'video_date': '2025-06-15'},
 {'video_id': 'GruRwbDbQ4Y',
  'video_title': 'Nu au micuții soluții 😂#georgesimion #unitate #democratie #impreuna #prosperitate #românia',
  'video_date': '2025-06-15'},
 {'video_id': 'DKhN-ua4lyw',
  'video_title': '#gs #georgesimion #democratie #impreuna #prosperitate #românia #unitate',
  'video_date': '2025-06-11'},
 {'video_id': '4WGWpctoAeQ',
  'video_title': '#georgesimion #unitate #democratie #impreuna #prosperitate #

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [8]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Turul României: realități de la firul ierbii


C:\Users\andre\AppData\Local\Temp\ipykernel_22100\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Episodul 2: Cum ne-au furat alegerile - Turismul Electoral
Colectez: Jurnalul video al zilei: Sâmbătă dimineață la Gorj în preajmă de Sânziene
Colectez: #georgesimion #unitate #democratie #impreuna #prosperitate #românia #gs
Colectez: Nu au micuții soluții 😂#georgesimion #unitate #democratie #impreuna #prosperitat
Colectez: #gs #georgesimion #democratie #impreuna #prosperitate #românia #unitate
Colectez: #georgesimion #unitate #democratie #impreuna #prosperitate #românia #gs
Colectez: #georgesimion #unitate #democratie #impreuna #prosperitate #românia #gs
Colectez: #unitate #democratie #georgesimion #impreuna #prosperitate #românia #gs
Colectez: Dumnezeu e mare! Explicăm situația din România prietenilor conservatori din Stat


100

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [9]:
comments[:3]

[{'id': 'yt_5rHoTX3U_3Q_UgwQCXWL6yLRE8S9TUV4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'georgesimionoficial',
  'text_raw': '🎉Felicitări și Respect Domnule George Simion și AUR!',
  'video_id': '5rHoTX3U_3Q',
  'video_title': 'Turul României: realități de la firul ierbii',
  'video_date': '2025-09-30',
  'comment_date': '2025-10-01',
  'likes': 18,
  'collected_at': '2026-05-12'},
 {'id': 'yt_5rHoTX3U_3Q_UgzKUz0vTNN_K_7vpyJ4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'georgesimionoficial',
  'text_raw': 'Respect domnule Simion!👍👏',
  'video_id': '5rHoTX3U_3Q',
  'video_title': 'Turul României: realități de la firul ierbii',
  'video_date': '2025-09-30',
  'comment_date': '2025-10-01',
  'likes': 6,
  'collected_at': '2026-05-12'},
 {'id': 'yt_5rHoTX3U_3Q_UgzPPFmNMk_Rvlxs43N4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'georgesimionoficial',
  'text_raw': 'RESPECT D.LE SIMION. DUCCES PANA LA CAPAT. DOAMNE AJUTA.',
  'video_id': '5rHoTX3

In [10]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [11]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [12]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_5rHoTX3U_3Q_UgwQCXWL6yLRE8S9TUV4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'georgesimionoficial',
 'text_raw': '🎉Felicitări și Respect Domnule George Simion și AUR!',
 'video_id': '5rHoTX3U_3Q',
 'video_title': 'Turul României: realități de la firul ierbii',
 'video_date': '2025-09-30',
 'comment_date': '2025-10-01',
 'likes': 18,
 'collected_at': '2026-05-12',
 'text': '🎉Felicitări și Respect Domnule George Simion și AUR!'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [13]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 100
Comentarii după filtrarea lungimii: 37


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [14]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 37


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [15]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 37


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [16]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 37
Fișier: c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\data\cleaned\student_02_youtube_clean.jsonl


# Functia de curatare

In [17]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [18]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 100
Comentarii curate: 37


In [19]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌.
CLEAN: Felicitării George Simion Președintele României!🇷🇴😇⛑️🇷🇴❤️🛐❤️✝️✝️✝️❤️🕯️💐🇷🇴🙌💯🙌.
---
RAW: Asa trebuie să fiți printre oameni nu sa se doarmă in parlament succes domnule Simion nu am greșit cind v-am votat
CLEAN: Asa trebuie să fiți printre oameni nu sa se doarmă in parlament succes domnule Simion nu am greșit cind v-am votat
---
RAW: Așa este, erau pregătiți să fraudeze în spatele a ceea ce se putea vedea. Am fost la secția de votare, nu a fost nimic ilegal vizibil, dar prea mulți votanți pe listele suplimentare! Pe tabletă nu se vedea acel beculeț roșu! Au avut o tehnică bine aranjată pentru a frauda!
CLEAN: Așa este, erau pregătiți să fraudeze în spatele a ceea ce se putea vedea. Am fost la secția de votare, nu a fost nimic ilegal vizibil, dar prea mulți votanți pe listele suplimentare! Pe tabletă nu se vedea acel beculeț roșu! Au avut o tehnică bine aranjată pentru a frauda!
---


In [20]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\andre\OneDrive\Desktop\ADC\ADC-A2S2\Inginerie_AI\echochamber-project-team-6\data\cleaned\student_02_youtube_clean.jsonl
Comentarii salvate: 37


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.